In [ ]:
# ==============================================================================
# KAGGLE SETUP BLOCK (INDEPENDENT BLOCK)
# ==============================================================================
import os
import sys
import subprocess
from pathlib import Path

# Detect Kaggle environment
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

if IS_KAGGLE:
    print("--- Detected Kaggle Environment ---")
    
    # 1. Clone repository if not present
    REPO_URL = "https://github.com/HCMUS-VIR-Nhom8/DINOv3-FAISS-HNSW-SOP-VisualProductSearch.git"
    REPO_DIR = Path("/kaggle/working/DINOv3-FAISS-HNSW-SOP-VisualProductSearch")
    
    if not REPO_DIR.exists():
        print(f"Cloning repository: {REPO_URL}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

    # swich to branch "experiments"
    print("Switching to branch: experiments")
    try:
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "experiments"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"Error switching branch: {e}. Attempting to fetch first...")
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "experiments"], check=True)
    
    # Change working directory of the kernel to notebooks directory
    # so that PROJECT_ROOT = Path("..").resolve() works correctly
    NOTEBOOKS_DIR = REPO_DIR / "notebooks"
    if NOTEBOOKS_DIR.exists():
        os.chdir(str(NOTEBOOKS_DIR))
        print(f"Changed working directory to: {os.getcwd()}")
        
    # Add project root to sys.path so src imports work
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
        print("Added repository root to python path.")
        
    # 2. Download and unzip dataset from Google Drive
    DATA_DIR = REPO_DIR / "data" / "raw" / "Stanford_Online_Products"
    ZIP_PATH = REPO_DIR / "Stanford_Online_Products.zip"
    
    if not DATA_DIR.exists() or not (DATA_DIR / "Ebay_train.txt").exists():
        print("Dataset not found. Downloading from Google Drive...")
        
        # Install gdown if needed
        try:
            import gdown
        except ImportError:
            print("Installing gdown...")
            subprocess.run([sys.executable, "-m", "pip", "install", "gdown"], check=True)
            import gdown
            
        # Download Stanford Online Products ZIP
        file_id = "1TclrpQOF_ullUP99wk_gjGN8pKvtErG8"
        url = f"https://drive.google.com/uc?id={file_id}"
        print(f"Downloading from Google Drive ID: {file_id}")
        gdown.download(url, str(ZIP_PATH), quiet=False)
        
        # Extract the zip file
        print("Extracting dataset...")
        import zipfile
        raw_dir = REPO_DIR / "data" / "raw"
        raw_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(str(ZIP_PATH), 'r') as zip_ref:
            zip_ref.extractall(str(raw_dir))
            
        print("Dataset extraction completed.")
        
        # Clean up zip file
        if ZIP_PATH.exists():
            ZIP_PATH.unlink()
            print("Cleaned up ZIP file.")
            
    # Note on HF login for gated DINOv3
    print("\n--- Hugging Face Access Note ---")
    print("DINOv3 is a gated Hugging Face model. If access token is required, run:")
    print("from huggingface_hub import login; login(token='YOUR_HF_TOKEN')\n")
else:
    print("Running in local environment. Setup skipped.")


# Notebook 03 — Proposed Method

## Mục tiêu

Notebook này hiện thực hóa **proposed method** cho bài toán Visual Product
Search trên Stanford Online Products (SOP), tiếp nối Notebook 01 (sampling)
và Notebook 02 (baseline):

```text
Gallery images
    -> preprocessing (ProposedPreprocessor)
    -> DINOv3 ([CLS] token)
    -> L2 normalization
    -> FAISS HNSW indexing
    -> save embeddings + index

Query image
    -> preprocessing (giống hệt gallery)
    -> DINOv3
    -> L2 normalization
    -> HNSW ANN search
    -> Top-M candidates
    -> metadata-based re-ranking (category consistency)
    -> Top-K
    -> evaluation + qualitative visualization
```

## Ràng buộc quan trọng — KHÔNG thay đổi baseline

Notebook 03 dùng lại **nguyên vẹn**:

- `data/sampled/baseline_gallery.csv` và `data/sampled/baseline_query.csv` do
  Notebook 02 tạo (không sampling lại, không tự tạo split khác).
- Cùng evaluation protocol: `Recall@1, Recall@5, Recall@10, Recall@20,
  Recall@50, Recall@100, mAP`, cùng cách đo latency (`encoding_*`,
  `search_*`, `end_to_end_*` tính bằng ms, mean/p50/p95) và cùng cách đo
  memory (MB) — để so sánh baseline vs proposed ở Notebook 04 là **fair**.

## Không "cải tiến" method

Notebook chỉ hiện thực đúng: **DINOv3 + L2 normalization + FAISS HNSW +
metadata re-ranking (category-consistency) + evaluation** — không thêm
augmentation, PCA, IVF/PQ, cross-encoder, neural reranker hay đổi backbone.

## Tái sử dụng code đã có trong repo

Notebook import trực tiếp từ `src/`:

```python
from src.preprocessing.pipeline import ProposedPreprocessor
from src.models.encoder import DINOv3Encoder
from src.retrieval.hnsw import HNSWRetriever
from src.reranking.metadata import category_consistency_rerank
```

thay vì viết lại các thành phần đã tồn tại. Các hàm đánh giá (Recall@K, AP)
được viết lại **giống hệt** Notebook 02 (thay vì `src/evaluation/metrics.py`,
vốn dùng key `recall@k` viết thường) để đảm bảo tên metric khớp 100% với
Notebook 02 — đây là lựa chọn có chủ đích cho tính nhất quán, không phải
duplicate code tùy tiện.


## 0. Configuration

### Cell 1 — Configuration

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

# ---------------------------------------------------------------------------
# Dataset — PHẢI dùng đúng split đã tạo ở Notebook 02 (baseline). Không
# sampling lại, không tự tạo query/gallery split khác, để so sánh baseline
# vs proposed là fair (cùng gallery, cùng query, cùng ground-truth).
# ---------------------------------------------------------------------------
SPLIT_DIR = PROJECT_ROOT / "data" / "sampled"
SAMPLE_FILE = SPLIT_DIR / "sop_20k.csv"
GALLERY_FILE = SPLIT_DIR / "baseline_gallery.csv"
QUERY_FILE = SPLIT_DIR / "baseline_query.csv"

# Output proposed
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "proposed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QUAL_DIR = OUTPUT_DIR / "qualitative"

EMBEDDING_FILE = OUTPUT_DIR / "gallery_embeddings.npy"
GALLERY_META_FILE = OUTPUT_DIR / "gallery_metadata.csv"
HNSW_INDEX_FILE = OUTPUT_DIR / "gallery_hnsw.index"
QUERY_RESULTS_FILE = OUTPUT_DIR / "retrieval_results.npy"
QUERY_SCORES_FILE = OUTPUT_DIR / "retrieval_scores.npy"
METRICS_FILE = OUTPUT_DIR / "metrics.json"
LATENCY_FILE = OUTPUT_DIR / "latency.json"
CONFIG_FILE = OUTPUT_DIR / "config.json"

# ---------------------------------------------------------------------------
# Model — DINOv3. Tên model lấy từ configs/proposed.yaml của repo, KHÔNG
# hard-code khác đi. DINOV3_LOCAL_PATH cho phép trỏ tới checkpoint cục bộ
# (thư mục chứa config.json/model weights kiểu HuggingFace) thay vì Hub.
# ---------------------------------------------------------------------------
DINOV3_MODEL_NAME = "facebook/dinov3-vitb16-pretrain-lvd1689m"  # configs/proposed.yaml -> model.name
DINOV3_LOCAL_PATH = None  # vd: "/path/to/local/dinov3-checkpoint"
MODEL_SOURCE = DINOV3_LOCAL_PATH if DINOV3_LOCAL_PATH else DINOV3_MODEL_NAME

# ---------------------------------------------------------------------------
# Preprocessing — theo đúng ProposedPreprocessor (src/preprocessing/pipeline.py)
# và giá trị mặc định trong configs/proposed.yaml. Localization/segmentation
# (Grounding DINO / SAM) KHÔNG được bật: tài liệu phương pháp (mục 4.1.2.3)
# mô tả đây là các thành phần "có thể thay thế" trong prototype, và repo
# hiện chưa gắn checkpoint thật cho hai bước này (CONFIGURABLE, không phải
# method chính thức bắt buộc).
# ---------------------------------------------------------------------------
RESIZE_LONG_SIDE = 1024
TARGET_SIZE = 518            # kích thước sau letterbox
IMAGE_SIZE = TARGET_SIZE     # alias cho đúng tên biến yêu cầu trong đề bài
PADDING_RATIO = 0.10          # chưa dùng vì localization/segmentation đang tắt
BLUR_THRESHOLD = 50.0
JPEG_THRESHOLD = 8.0
USE_ILLUMINATION_CORRECTION = True
CLAHE_CLIP_LIMIT = 2.0

BATCH_SIZE = 16
NUM_WORKERS = 2

# ---------------------------------------------------------------------------
# FAISS HNSW — mặc định lấy từ configs/proposed.yaml (mục 4.1.7 trong tài
# liệu phương pháp). Đổi ở đây để thử nghiệm đánh đổi recall/latency/memory.
# ---------------------------------------------------------------------------
HNSW_M = 32
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH = 64

# ---------------------------------------------------------------------------
# Metadata re-ranking — dùng ĐÚNG công thức đã hiện thực trong
# src/reranking/metadata.py::category_consistency_rerank (không tự bịa công
# thức khác — xem mục 14 của yêu cầu).
#
# LƯU Ý QUAN TRỌNG (xem README của repo): SOP không có metadata dạng
# title/brand/mô tả văn bản cho ảnh truy vấn. Do đó công thức này KHÔNG so
# khớp metadata của query với candidate (không phải oracle match), mà tính
# điểm đồng thuận (consensus) super_class_id ngay trong tập Top-N candidate:
#
#     meta_score(candidate) = tần suất super_class_id của candidate đó
#                              trong Top-N candidates / N
#     final_score = alpha_visual * visual_score_normalized
#                   + (1 - alpha_visual) * meta_score
#
# Đây là một CONFIGURABLE EXPERIMENTAL CHOICE đã được README của repo xác
# nhận, KHÔNG PHẢI một công thức chính thức đã được đề tài chứng minh tối
# ưu. Nếu có metadata text/brand thật, hãy thay hàm rerank tương ứng thay vì
# coi category-consistency là mặc định vĩnh viễn.
# ---------------------------------------------------------------------------
ENABLE_RERANKING = True
RERANK_CANDIDATES = 100      # = retrieval.candidate_k trong configs/proposed.yaml
ALPHA_VISUAL = 0.85          # = reranking.alpha_visual trong configs/proposed.yaml
VISUAL_WEIGHT = ALPHA_VISUAL
METADATA_WEIGHT = 1.0 - ALPHA_VISUAL

# ---------------------------------------------------------------------------
# Evaluation — PHẢI trùng KS đã dùng ở Notebook 02 để so sánh công bằng.
# ---------------------------------------------------------------------------
KS = [1, 5, 10, 20, 50, 100]
MAX_QUERIES = None           # None = toàn bộ query set
RANDOM_SEED = 42

# Device — đổi thành "cpu" nếu không có GPU.
DEVICE = "cuda"

N_QUALITATIVE = 20            # số lượng ảnh minh họa xuất ra qualitative/

print("PROJECT_ROOT :", PROJECT_ROOT)
print("GALLERY_FILE :", GALLERY_FILE)
print("QUERY_FILE   :", QUERY_FILE)
print("OUTPUT_DIR   :", OUTPUT_DIR)
print("MODEL_SOURCE :", MODEL_SOURCE)

## 1. Environment & Imports

### Cell 2 — Imports & kiểm tra môi trường

In [ ]:
%pip install faiss-gpu

In [ ]:
import os
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

import torch
import transformers

try:
    import faiss
except ImportError as e:
    raise ImportError(
        "Không import được faiss. Cài đặt bằng:\n"
        "  pip install faiss-cpu   # hoặc faiss-gpu nếu có CUDA phù hợp\n"
        f"Lỗi gốc: {e}"
    )

# Cho phép "from src...." hoạt động khi notebook chạy từ thư mục notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("FAISS       :", getattr(faiss, "__version__", "n/a"))
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
elif DEVICE == "cuda":
    print(
        "\u26a0 DEVICE='cuda' nhưng không có GPU khả dụng. "
        "Hãy đổi DEVICE='cpu' ở Cell Configuration trước khi chạy tiếp."
    )

### Cell 3 — Import module đã có trong repo (không duplicate)

In [ ]:
# Theo mục 23 của yêu cầu: dùng lại module đã có trong src/ thay vì viết lại.
try:
    from src.preprocessing.pipeline import ProposedPreprocessor
    from src.models.encoder import DINOv3Encoder
    from src.retrieval.hnsw import HNSWRetriever
    from src.reranking.metadata import category_consistency_rerank
except ImportError as e:
    raise ImportError(
        "Không import được các module trong src/. Kiểm tra:\n"
        "  1. Notebook đang chạy từ thư mục notebooks/ (PROJECT_ROOT = ..)\n"
        "  2. Repo có đầy đủ src/preprocessing/pipeline.py, src/models/encoder.py, "
        "src/retrieval/hnsw.py, src/reranking/metadata.py\n"
        f"Lỗi gốc: {e}"
    )

print("\u2713 Đã import ProposedPreprocessor, DINOv3Encoder, HNSWRetriever, "
      "category_consistency_rerank từ src/.")

## 2. Load Dataset Split

### Cell 4 — Kiểm tra input từ Notebook 01/02

In [ ]:
if not (GALLERY_FILE.exists() and QUERY_FILE.exists()):
    print('Baseline split files not found. Generating split from sop_20k.csv...')
    if not SAMPLE_FILE.exists():
        raise FileNotFoundError(f'Cannot find sample file: {SAMPLE_FILE}')
    sample_df = pd.read_csv(SAMPLE_FILE)
    gallery_parts = []
    query_parts = []
    for class_id, group in sample_df.groupby('class_id'):
        group = group.sample(frac=1, random_state=RANDOM_SEED + int(class_id) % 100000)
        n = len(group)
        n_query = max(1, int(round(n * 0.20)))
        n_query = min(n_query, n - 1)
        query_parts.append(group.iloc[:n_query])
        gallery_parts.append(group.iloc[n_query:])
    query_df = pd.concat(query_parts, ignore_index=True)
    gallery_df = pd.concat(gallery_parts, ignore_index=True)
    gallery_df.to_csv(GALLERY_FILE, index=False)
    query_df.to_csv(QUERY_FILE, index=False)
    print(f'Generated and saved: {GALLERY_FILE} and {QUERY_FILE}')
else:
    print('✓ Found existing GALLERY_FILE and QUERY_FILE.')


### Cell 5 — Load gallery/query split (giống hệt Notebook 02)

In [ ]:
gallery_df = pd.read_csv(GALLERY_FILE).reset_index(drop=True)
query_df = pd.read_csv(QUERY_FILE).reset_index(drop=True)

print("Gallery:", len(gallery_df), "| classes:", gallery_df["class_id"].nunique())
print("Query  :", len(query_df), "| classes:", query_df["class_id"].nunique())

display(gallery_df.head())

### Cell 6 — Kiểm tra tính nhất quán của split

In [ ]:
required_columns = {"image_id", "class_id", "image_path"}

for name, df in [("gallery_df", gallery_df), ("query_df", query_df)]:
    missing_cols = required_columns - set(df.columns)
    if missing_cols:
        raise ValueError(f"{name} thiếu cột bắt buộc: {missing_cols}")

# Mỗi query class phải có ít nhất 1 ảnh trong gallery (positive tồn tại)
missing_query_classes = set(query_df["class_id"]) - set(gallery_df["class_id"])
assert len(missing_query_classes) == 0, (
    f"{len(missing_query_classes)} class trong query không có ảnh gallery tương ứng."
)

# Query và gallery không được trùng image_id (tránh self-retrieval ảo)
overlap = set(query_df["image_id"]) & set(gallery_df["image_id"])
assert len(overlap) == 0, f"Query và gallery bị overlap {len(overlap)} image_id."

# Kiểm tra image path tồn tại trên đĩa
for name, df in [("gallery_df", gallery_df), ("query_df", query_df)]:
    exists = df["image_path"].map(os.path.exists)
    if not exists.all():
        n_missing = int((~exists).sum())
        raise FileNotFoundError(f"{name} có {n_missing} image_path không tồn tại trên đĩa.")

# Reranking cần super_class_id — nếu thiếu, báo lỗi rõ ràng thay vì âm thầm tắt.
if ENABLE_RERANKING and "super_class_id" not in gallery_df.columns:
    raise ValueError(
        "ENABLE_RERANKING=True nhưng gallery_df không có cột 'super_class_id'.\n"
        "category_consistency_rerank() cần cột này để tính metadata score.\n"
        "Hãy đảm bảo Notebook 01 giữ lại cột super_class_id khi tạo sop_20k.csv, "
        "hoặc đặt ENABLE_RERANKING = False ở Cell Configuration để chạy HNSW-only."
    )

print(
    "\u2713 Split hợp lệ: không overlap, mọi query class đều có gallery, "
    "mọi image_path tồn tại"
    + (", có super_class_id cho reranking." if ENABLE_RERANKING else ".")
)

## 3. Load DINOv3

`DINOv3Encoder` (`src/models/encoder.py`) tải processor + model qua
HuggingFace Transformers (`AutoImageProcessor`, `AutoModel`), đưa model lên
GPU nếu có và gọi `.eval()`.

**Lưu ý:** `facebook/dinov3-vitb16-pretrain-lvd1689m` là model có kiểm soát
truy cập (gated) trên Hugging Face Hub. Trước khi chạy cell dưới, hãy:

```bash
huggingface-cli login
```

và đảm bảo tài khoản đã được cấp quyền truy cập model tại trang model card
trên huggingface.co.

### Cell 7 — Load DINOv3 processor + model

In [ ]:
try:
    dinov3 = DINOv3Encoder(MODEL_SOURCE, DEVICE)
except Exception as e:
    raise RuntimeError(
        f"Không load được DINOv3 ({MODEL_SOURCE}).\n"
        "Nguyên nhân thường gặp:\n"
        "  1. Chưa đăng nhập Hugging Face hoặc chưa được cấp quyền truy cập "
        "model gated -> chạy `huggingface-cli login` và xin quyền tại trang "
        "model card trên huggingface.co.\n"
        "  2. Không có kết nối mạng tới huggingface.co.\n"
        "  3. DINOV3_LOCAL_PATH (Cell Configuration) trỏ sai đường dẫn checkpoint cục bộ.\n"
        f"Lỗi gốc: {e}"
    )

print("\u2713 Đã load DINOv3Encoder")
print("Device       :", dinov3.device)
print("Embedding dim:", dinov3.dim)

### Cell 8 — Đưa model lên GPU (nếu có), eval() và freeze parameters

In [ ]:
# LƯU Ý: DINOv3Encoder.__init__ (src/models/encoder.py) hiện chỉ gọi
# .to(device).eval() và CHƯA tự set requires_grad_(False) cho tham số.
# Để đúng yêu cầu "frozen backbone" của tài liệu phương pháp (mục 4.1.3.3:
# "Có thể sử dụng như một backbone đóng băng"), notebook tự đóng băng tường
# minh ở đây.
# TODO (repo): cân nhắc chuyển đoạn freeze này vào DINOv3Encoder.__init__.
for p in dinov3.model.parameters():
    p.requires_grad_(False)

num_params = sum(p.numel() for p in dinov3.model.parameters())
num_trainable = sum(p.numel() for p in dinov3.model.parameters() if p.requires_grad)

print("Tổng số tham số     :", f"{num_params:,}")
print("Tham số có thể train:", f"{num_trainable:,}")
print("model.training =", dinov3.model.training)

assert num_trainable == 0, "DINOv3 backbone chưa được đóng băng hoàn toàn."
assert dinov3.model.training is False, "DINOv3 model chưa ở chế độ eval()."

print("\u2713 Backbone đã đóng băng và ở chế độ eval — đúng yêu cầu frozen backbone.")

### Cell 9 — Inspect DINOv3 output trên 1 ảnh thật

In [ ]:
sample_row = gallery_df.iloc[0]
sample_image = Image.open(sample_row["image_path"]).convert("RGB")

inputs = dinov3.processor(images=[sample_image], return_tensors="pt").to(dinov3.device)
with torch.inference_mode():
    raw_outputs = dinov3.model(**inputs)

print("Kiểu output của DINOv3.model():", type(raw_outputs))
print("last_hidden_state.shape:", tuple(raw_outputs.last_hidden_state.shape))
print(
    "  -> [batch, num_tokens, hidden_dim]. Token đầu tiên (index 0) là "
    "global/[CLS] token, dùng làm global embedding (Hình 4.6 trong tài liệu)."
)

### Cell 10 — Hàm dinov3_to_embedding (đối chiếu với DINOv3Encoder.encode)

In [ ]:
def dinov3_to_embedding(outputs):
    """
    Chuyển output token-level của DINOv3 (last_hidden_state: [B, T, D]) thành
    một global embedding [B, D] bằng CLS token (last_hidden_state[:, 0]).

    Đây CHÍNH LÀ pooling mà DINOv3Encoder.encode() (src/models/encoder.py) đã
    dùng — hàm này chỉ để minh họa/kiểm chứng lại pooling, KHÔNG dùng để
    encode thật (encode thật dùng dinov3.encode(images) ở các cell sau, để
    không duplicate logic).

    Nếu tài liệu phương pháp quy định pooling khác (vd. mean-pooling patch
    token), hãy đổi hàm này — hiện tại repo/README chỉ xác nhận CLS token,
    nên đây được đánh dấu là OFFICIAL METHOD theo đúng những gì repo đã cài.
    """
    return outputs.last_hidden_state[:, 0]


manual_cls = dinov3_to_embedding(raw_outputs)
manual_cls_normalized = torch.nn.functional.normalize(manual_cls, p=2, dim=1)
manual_cls_np = manual_cls_normalized.cpu().numpy().astype("float32")

official_embedding = dinov3.encode([sample_image])  # đã L2-normalize bên trong

print("manual CLS embedding shape :", manual_cls_np.shape)
print("dinov3.encode() shape      :", official_embedding.shape)

assert np.allclose(manual_cls_np, official_embedding, atol=1e-4), (
    "dinov3_to_embedding() không khớp với DINOv3Encoder.encode() — kiểm tra lại pooling."
)

print("\u2713 Pooling CLS token khớp giữa hàm minh họa và DINOv3Encoder.encode().")

## 4. DINOv3 Embedding Extraction

### Cell 11 — Hàm L2 normalization

In [ ]:
def l2_normalize(x, eps=1e-12):
    """Row-wise L2 normalization cho numpy array [N, D]."""
    norm = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(norm, eps, None)


test_x = np.random.randn(4, dinov3.dim).astype("float32")
test_y = l2_normalize(test_x)
norms = np.linalg.norm(test_y, axis=1)
assert np.allclose(norms, 1.0, atol=1e-5)
print(
    "\u2713 l2_normalize() hoạt động đúng (dùng để double-check; "
    "DINOv3Encoder.encode() đã tự L2-normalize bên trong bằng F.normalize)."
)

### Preprocessing đề xuất (`ProposedPreprocessor`)

Tái sử dụng `ProposedPreprocessor` (`src/preprocessing/pipeline.py`), hiện
thực đúng phần đã tái lập được của chuỗi bước trong mục 4.1.2 của tài liệu:

```text
Orientation -> Resize (giữ tỷ lệ) -> Quality flags (không loại ảnh)
            -> Letterbox -> Illumination correction (CLAHE)
```

Localization/Segmentation (Grounding DINO / SAM) KHÔNG được bật — tài liệu
mô tả đây là các thành phần "có thể thay thế" trong prototype (mục 4.1.2.3),
và repo chưa gắn checkpoint thật cho hai bước này. Không tự ý thêm
augmentation/crop/flip/color-jitter vì đây là retrieval evaluation.

### Cell 12 — Khởi tạo ProposedPreprocessor + preview

In [ ]:
proposed_preprocessor = ProposedPreprocessor(
    resize_long_side=RESIZE_LONG_SIDE,
    target_size=TARGET_SIZE,
    padding_ratio=PADDING_RATIO,
    blur_threshold=BLUR_THRESHOLD,
    jpeg_threshold=JPEG_THRESHOLD,
    use_illumination=USE_ILLUMINATION_CORRECTION,
    clahe_clip_limit=CLAHE_CLIP_LIMIT,
)

preview_row = gallery_df.iloc[0]
preview_original = Image.open(preview_row["image_path"]).convert("RGB")
preview_output = proposed_preprocessor(preview_row["image_path"])

plt.figure(figsize=(9, 4))
ax = plt.subplot(1, 2, 1)
ax.imshow(preview_original)
ax.set_title("Original")
ax.axis("off")

ax = plt.subplot(1, 2, 2)
ax.imshow(preview_output.image)
ax.set_title(f"Proposed preprocessing: {TARGET_SIZE}\u00d7{TARGET_SIZE}")
ax.axis("off")
plt.tight_layout()
plt.show()

print("blur_score:", preview_output.blur_score, "| flags:", preview_output.metadata)
print(
    "Lưu ý: AutoImageProcessor của HuggingFace sẽ áp dụng thêm resize/normalize "
    "riêng của DINOv3 lên ảnh đã letterbox này khi gọi dinov3.encode() — đây là "
    "thiết kế sẵn có của repo (DINOv3Encoder), không phải augmentation bổ sung "
    "do notebook này thêm vào."
)

### Hàm extract embeddings (offline batch inference)

Không dùng `torch.utils.data.DataLoader` như baseline: `DINOv3Encoder.encode()`
nhận trực tiếp một list ảnh PIL (tự batch qua `AutoImageProcessor` bên trong,
đã bọc `torch.inference_mode()`), nên ở đây chỉ cần chia `DataFrame` thành
các mini-batch theo `BATCH_SIZE` thay vì một `Dataset`/`DataLoader` kiểu
torchvision như ở Notebook 02.

### Cell 13 — Định nghĩa extract_dinov3_embeddings()

In [ ]:
def extract_dinov3_embeddings(df, preprocessor, encoder, batch_size, desc="Encoding"):
    """
    Trích xuất embedding DINOv3 cho toàn bộ DataFrame (offline, batch inference).

    - torch.inference_mode() + model.eval(): đã cài đặt sẵn bên trong
      DINOv3Encoder.encode() (src/models/encoder.py).
    - GPU support / CPU fallback: theo dinov3.device (đã resolve ở Cell load
      DINOv3Encoder).
    - Trả về numpy float32, đã L2-normalize.

    Trả về:
        embeddings   : np.ndarray [N, D] float32, đã L2-normalize
        image_ids    : np.ndarray [N]
        class_ids    : np.ndarray [N]
        quality_rows : list[dict]  (blur_score, jpeg_score, flags mỗi ảnh)
        elapsed      : float (giây)
    """
    all_embeddings = []
    all_image_ids = []
    all_class_ids = []
    quality_rows = []

    start = time.perf_counter()

    for i in tqdm(range(0, len(df), batch_size), desc=desc):
        chunk = df.iloc[i:i + batch_size]
        images = []
        for _, row in chunk.iterrows():
            out = preprocessor(row["image_path"])
            images.append(out.image)
            quality_rows.append({
                "image_id": int(row["image_id"]),
                "blur_score": out.blur_score,
                "jpeg_score": out.jpeg_score,
                **(out.metadata or {}),
            })

        batch_embeddings = encoder.encode(images)
        all_embeddings.append(batch_embeddings)
        all_image_ids.extend(chunk["image_id"].tolist())
        all_class_ids.extend(chunk["class_id"].tolist())

    elapsed = time.perf_counter() - start
    embeddings = np.concatenate(all_embeddings, axis=0).astype("float32")

    return (
        embeddings,
        np.asarray(all_image_ids),
        np.asarray(all_class_ids),
        quality_rows,
        elapsed,
    )

## 5. Offline Gallery Feature Extraction

### Cell 14 — Offline: extract gallery embeddings

In [ ]:
if DEVICE == "cuda" and torch.cuda.is_available():
    torch.cuda.synchronize()

gallery_embeddings, gallery_image_ids, gallery_class_ids, gallery_quality, gallery_encode_time = \
    extract_dinov3_embeddings(
        gallery_df, proposed_preprocessor, dinov3, BATCH_SIZE, desc="Gallery embeddings"
    )

if DEVICE == "cuda" and torch.cuda.is_available():
    torch.cuda.synchronize()

print("Embedding shape:", gallery_embeddings.shape)
print("Extraction time:", gallery_encode_time, "sec")
print("dtype:", gallery_embeddings.dtype)

### Cell 15 — Validate gallery embeddings

In [ ]:
assert gallery_embeddings.ndim == 2
assert gallery_embeddings.shape[0] == len(gallery_df), (
    "Số embedding không khớp số ảnh gallery — kiểm tra lại bước encode."
)
assert gallery_embeddings.shape[1] == dinov3.dim

embedding_norms = np.linalg.norm(gallery_embeddings, axis=1)
print("Norm min :", embedding_norms.min())
print("Norm max :", embedding_norms.max())
print("Norm mean:", embedding_norms.mean())

assert np.allclose(embedding_norms, 1.0, atol=1e-4)
print("\u2713 Gallery embeddings (DINOv3) đã L2-normalized.")

### Cell 16 — Lưu gallery embeddings + metadata

In [ ]:
np.save(EMBEDDING_FILE, gallery_embeddings)

quality_df = pd.DataFrame(gallery_quality)
gallery_meta = gallery_df.merge(quality_df, on="image_id", how="left")
gallery_meta.to_csv(GALLERY_META_FILE, index=False)

print("Embedding saved:", EMBEDDING_FILE)
print("Metadata saved :", GALLERY_META_FILE)

### Cell 17 — Đo memory của gallery embedding

In [ ]:
gallery_embedding_mb = gallery_embeddings.nbytes / (1024 ** 2)

print(f"Gallery embedding memory: {gallery_embedding_mb:.2f} MB")
print(
    f"Per vector: {gallery_embeddings.shape[1] * 4 / 1024:.2f} KB "
    f"(feature_dim={gallery_embeddings.shape[1]}, float32)"
)

## 6. Build FAISS HNSW Index

Vì embedding đã L2-normalize, cosine similarity = inner product:
`cosine(q, x) = q · x`. `HNSWRetriever` (`src/retrieval/hnsw.py`) dùng
`faiss.IndexHNSWFlat(dim, M, faiss.METRIC_INNER_PRODUCT)`, KHÔNG nén vector
(Flat) — giữ nguyên độ chính xác của similarity search, đúng lựa chọn đã nêu
ở mục 4.1.5 của tài liệu phương pháp.

Tham số quan trọng (cấu hình ở Cell Configuration): `HNSW_M`,
`HNSW_EF_CONSTRUCTION`, `HNSW_EF_SEARCH`.

### Cell 18 — Tạo HNSW index, add vectors, đo build time

In [ ]:
hnsw_retriever = HNSWRetriever(
    dim=gallery_embeddings.shape[1],
    M=HNSW_M,
    ef_construction=HNSW_EF_CONSTRUCTION,
    ef_search=HNSW_EF_SEARCH,
)

build_start = time.perf_counter()
hnsw_retriever.add(gallery_embeddings)
hnsw_build_time = time.perf_counter() - build_start

print(f"HNSW build time: {hnsw_build_time:.4f} sec")
print("ntotal:", hnsw_retriever.index.ntotal)

assert hnsw_retriever.index.ntotal == len(gallery_df), (
    "HNSW ntotal khác số lượng ảnh trong gallery — kiểm tra lại bước add()."
)
print("\u2713 HNSW ntotal khớp với kích thước gallery.")

### Cell 19 — Sanity check: self-retrieval trên vài vector gallery

In [ ]:
test_scores, test_ids = hnsw_retriever.search(gallery_embeddings[:5], k=1)
self_hit = (test_ids[:, 0] == np.arange(5)).mean()

print("Top-1 self-retrieval trên 5 vector gallery đầu tiên:", test_ids[:, 0])
print(f"Self-hit rate: {self_hit:.2f} (kỳ vọng gần 1.0 vì HNSW là approximate)")

### Cell 20 — Lưu HNSW index + đo dung lượng file

In [ ]:
hnsw_retriever.save(HNSW_INDEX_FILE)
hnsw_index_mb = HNSW_INDEX_FILE.stat().st_size / (1024 ** 2)

print("Index saved:", HNSW_INDEX_FILE)
print(f"HNSW index size: {hnsw_index_mb:.2f} MB")

## 7. Save Offline Artifacts

### Offline → Online boundary

Offline đã hoàn thành:

```text
Gallery images -> preprocessing -> DINOv3 -> L2 -> gallery_embeddings.npy
                                                  -> gallery_hnsw.index
```

Online chỉ cần: Query image -> preprocessing -> DINOv3 -> L2 -> HNSW search
trên index đã build -> (tuỳ chọn) metadata re-ranking -> Top-K.

**Không chạy lại DINOv3 cho gallery.**

# ONLINE PHASE

## 8. Query Encoding

### Cell 21 — Hàm encode_query()

In [ ]:
# ==============================================================================
# KAGGLE OUTPUT & RESULTS PREVIEW BLOCK (INDEPENDENT BLOCK)
# ==============================================================================
import os
import shutil
from pathlib import Path
import pandas as pd

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

if IS_KAGGLE:
    print("--- Kaggle Output & Preview ---")
    
    # 1. Print preview
    print("Preview of offline proposed embeddings and index:")
    embedding_file = Path("outputs/proposed/gallery_embeddings.npy")
    if embedding_file.exists():
        import numpy as np
        emb = np.load(embedding_file)
        print(f"Gallery proposed embeddings shape: {emb.shape}")
    index_file = Path("outputs/proposed/gallery_hnsw.index")
    if index_file.exists():
        print(f"HNSW index file size: {index_file.stat().st_size / (1024**2):.2f} MB")
        
    # 2. Export to /kaggle/working
    print("\nCopying results to Kaggle output directory (/kaggle/working)...")
    for folder in ["outputs/proposed", "data/sampled"]:
        dest = Path("/kaggle/working") / folder
        dest.mkdir(parents=True, exist_ok=True)
        src = Path(folder)
        if src.exists():
            for item in src.iterdir():
                if item.is_file():
                    shutil.copy(item, dest)
                    print(f"Copied {item.name} to {dest}")
